# 1. Imports

In [1]:
import os
import weave
import wandb
import torch
import pandas as pd
import os.path as osp
import io
import base64
import re
import trl
import inspect
import time
import numpy as np
from openai import OpenAI
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from models.mllm_wrappers import LLaVA, Qwen, Janus, Qwen3_5, Molmo
from peft import LoraConfig, get_peft_model
from configuration_template import Config
from bert_score import BERTScorer
from data_prep.data_utils import build_grid_image, build_patch_image_with_target_highlight
from trl import GRPOConfig, GRPOTrainer
from sentence_transformers import SentenceTransformer, util




/home/alesau/miniconda3/envs/mllm/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
os.environ["WANDB_API_KEY"] = "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"
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/alesau/.netrc
wandb: Currently logged in as: alex-esau98 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
#!pip install -U transformers accelerate datasets peft trl qwen-vl-utils wandb weave bitsandbytes
#!pip install wandb
#!pip install weave

: 

: 

# 2. Inits

### Patches

In [3]:


weave.init('alex-esau98/mllm')


wandb.init(
    entity="alex-esau98",
    project="mllm",
    config={
        "model": "Qwen/Qwen2-VL-2B-Instruct",
        "method": "GRPO",
        "reward": "judge_accuracy_plus_penalties"
    }
)

weave: weave version 0.53.11 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: alex-esau98.
weave: View Weave data at https://wandb.ai/alex-esau98/mllm/weave


wandb: Initializing weave.


In [4]:
config = Config(model="Qwen/Qwen2-VL-2B-Instruct")
config.input_type = 'simple'
config.task = 'patches'

img_kwargs = {
        "patch_size": config.patch_size,
        "patch_padding": config.patch_padding,
        "patch_pad_color": config.patch_pad_color,
    }

generate_kwargs = {
        "max_new_tokens": config.max_new_tokens,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "min_p":0.0,
        "presence_penalty": 1.5,
        "repetition_penalty": 1.0,
    }

model_kwargs = {"cache_dir": config.model_cache_dir, "quant": config.quant}

Model = Qwen
model = Model(config.model_id, **model_kwargs)
print(f'Model: {model.__class__}')

model_version: Qwen2
model_type: Qwen
/home/alesau/mllm/mllm/PATH
building Qwen model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model: <class 'models.mllm_wrappers.Qwen'>


In [5]:
config_judge = Config(model="Qwen/Qwen3.5-4B")
config_judge.input_type = 'simple'
config_judge.task = 'patches'

img_kwargs_judge = {
        "patch_size": config.patch_size,
        "patch_padding": config.patch_padding,
        "grid_padding": config.grid_padding,
        "patch_pad_color": config.patch_pad_color,
        "grid_pad_color": config.grid_pad_color,
    }

generate_kwargs_judge = {
        "max_new_tokens": config_judge.max_new_tokens,
        "do_sample": True,
        "temperature": 1.0,
        "top_p": 0.85,
        "top_k": 20,
        "min_p":0.0,
        "presence_penalty": 1.5,
        "repetition_penalty": 1.0,
    }       

model_kwargs_judge = {"cache_dir": config_judge.model_cache_dir, "quant": config_judge.quant}

JudgeModel = Qwen3_5
judge_model = JudgeModel(config_judge.model_id, **model_kwargs_judge)
print(f'Judge Model: {judge_model.__class__}')

#judge_model = OpenAI(
#    api_key="EMPTY",
#    base_url="http://0.0.0.0:8080/v1",
#)


model_version: Qwen3.5
model_type: Qwen3_5
/home/alesau/mllm/mllm/PATH
building Qwen3_5 model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Judge Model: <class 'models.mllm_wrappers.Qwen3_5'>


In [5]:

print(type(model))
print(model.__dict__.keys())
print(type(model.model))
print(type(model.processor))

print()
print(type(judge_model))
print(judge_model.__dict__.keys())
print(type(judge_model.model))
print(type(judge_model.processor))
print(judge_model.model_size)

<class 'models.mllm_wrappers.Qwen'>
dict_keys(['model', 'processor', 'model_id', 'model_size', 'quant', 'device'])
<class 'transformers.models.qwen2_vl.modeling_qwen2_vl.Qwen2VLForConditionalGeneration'>
<class 'transformers.models.qwen2_vl.processing_qwen2_vl.Qwen2VLProcessor'>

<class 'models.mllm_wrappers.Qwen3_5'>
dict_keys(['model', 'processor', 'model_id', 'model_size', 'quant', 'device'])
<class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForConditionalGeneration'>
<class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>
4b


In [6]:


hf_model = model.model
processor = model.processor

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

hf_model = get_peft_model(hf_model, lora_config)

hf_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 2,213,343,744 || trainable%: 0.1969


### Grids

In [3]:


weave.init('alex-esau98/mllm')


wandb.init(
    entity="alex-esau98",
    project="mllm",
    config={
        "model": "Qwen/Qwen2-VL-2B-Instruct",
        "method": "GRPO",
        "reward": "judge_accuracy_plus_penalties"
    }
)

weave: weave version 0.53.7 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: alex-esau98.
weave: View Weave data at https://wandb.ai/alex-esau98/mllm/weave


wandb: Initializing weave.


In [4]:
config = Config(model="Qwen/Qwen2-VL-2B-Instruct")
config.input_type = 'simple'
config.task = 'grid'

img_kwargs = {
        "patch_size": config.patch_size,
        "patch_padding": config.patch_padding,
        "grid_padding": config.grid_padding,
        "patch_pad_color": config.patch_pad_color,
        "grid_pad_color": config.grid_pad_color,
    }

generate_kwargs = {
        "max_new_tokens": config.max_new_tokens,
        "do_sample": True,
        "temperature": 0.8
    }

model_kwargs = {"cache_dir": config.model_cache_dir, "quant": config.quant}

Model = Qwen
model = Model(config.model_id, **model_kwargs)
print(f'Model: {model.__class__}')

model_version: Qwen2
model_type: Qwen
/home/alesau/mllm/mllm/PATH
building Qwen model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model: <class 'models.mllm_wrappers.Qwen'>


In [5]:
config_judge = Config(model="Qwen/Qwen3.5-9B")
config_judge.input_type = 'simple'
config_judge.task = 'grid'

img_kwargs_judge = {
        "patch_size": config.patch_size,
        "patch_padding": config.patch_padding,
        "grid_padding": config.grid_padding,
        "patch_pad_color": config.patch_pad_color,
        "grid_pad_color": config.grid_pad_color,
    }

generate_kwargs_judge = {
        "max_new_tokens": config_judge.max_new_tokens,
        "do_sample": True,
        "temperature": 0.8
    }       

model_kwargs_judge = {"cache_dir": config_judge.model_cache_dir, "quant": config_judge.quant}

#JudgeModel = Qwen3_5
#judge_model = JudgeModel(config_judge.model_id, **model_kwargs_judge)
#print(f'Judge Model: {judge_model.__class__}')

from openai import OpenAI

judge_model = OpenAI(
    api_key="EMPTY",
    base_url="http://0.0.0.0:8080/v1",
)


model_version: Qwen3.5
model_type: Qwen3_5
/home/alesau/mllm/mllm/PATH


In [7]:

print(type(model))
print(model.__dict__.keys())
print(type(model.model))
print(type(model.processor))

print()
print(type(judge_model))
print(judge_model.__dict__.keys())
print(type(judge_model.model))
print(type(judge_model.processor))
print(judge_model.model_size)

<class 'models.mllm_wrappers.Qwen'>
dict_keys(['model', 'processor', 'model_id', 'model_size', 'quant', 'device'])
<class 'transformers.models.qwen2_vl.modeling_qwen2_vl.Qwen2VLForConditionalGeneration'>
<class 'transformers.models.qwen2_vl.processing_qwen2_vl.Qwen2VLProcessor'>

<class 'models.mllm_wrappers.Qwen3_5'>
dict_keys(['model', 'processor', 'model_id', 'model_size', 'quant', 'device'])
<class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForConditionalGeneration'>
<class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>
9b


In [6]:


hf_model = model.model
processor = model.processor

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

hf_model = get_peft_model(hf_model, lora_config)

hf_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 2,213,343,744 || trainable%: 0.1969


# 3. Data Prep

### Patches

In [9]:
data_df = pd.read_json(osp.join(config.data_dir, "color_patch_data_far.json"))
print(len(data_df))
dataset_train = data_df.sample(frac=0.12665, random_state=1998)  # far = frac=0.12665; close=0.12865; split = 0.12735; general=0.04252
dataset_judge = data_df.drop(dataset_train.index)

print(len(dataset_train))
print(len(dataset_judge))
#data_df

#print(dataset_train)
print(f"Length of dataset_train: {len(dataset_train)}")
print(f"Share of split_condition: {len(dataset_train[dataset_train['condition'] == "split"]) / len(dataset_train)}")
print(f"Share of far_condition: {len(dataset_train[dataset_train['condition'] == "far"]) / len(dataset_train)}")
print(f"Share of close_condition: {len(dataset_train[dataset_train['condition'] == "close"]) / len(dataset_train)}")
#for i in range(70,80):
#    print(data_df["conversation"][i])

15791
2000
13791
Length of dataset_train: 2000
Share of split_condition: 0.0
Share of far_condition: 1.0
Share of close_condition: 0.0


### Grids

In [7]:

data_df = pd.read_json(osp.join(config.data_dir, "color_grid_data_far.json"))
print(len(data_df))
dataset_train = data_df.sample(frac=0.5502, random_state=42) ##grid = 0.55  patches = frac=0.12665
dataset_judge = data_df.drop(dataset_train.index)

print(len(dataset_train))
print(len(dataset_judge))
data_df

#for i in range(70,80):
#    print(data_df["conversation"][i])

3635
2000
1635


,gameid,game_idx,roundNum,round_id,condition,success,utterances,n_utterances,objs,target,speaker_order,listener_order,listener_clicked
0,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,1,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_1,FAR,True,"[[speaker, green top left corner, 1528144087280]]",1,"[{'shapes': [{'color': [322, 21, 50]}, {'color...",1,"[2, 1, 0]","[1, 0, 2]",0
1,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,4,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_4,FAR,True,"[[speaker, green in the middle square top row,...",1,"[{'shapes': [{'color': [294, 7, 50]}, {'color'...",0,"[1, 2, 0]","[2, 1, 0]",2
2,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,11,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_11,FAR,True,"[[speaker, bottom left not yellow not orange, ...",1,"[{'shapes': [{'color': [339, 87, 50]}, {'color...",1,"[1, 2, 0]","[0, 2, 1]",2
3,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,12,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_12,FAR,True,"[[speaker, top middle is grey, 1528144497114]]",1,"[{'shapes': [{'color': [199, 26, 50]}, {'color...",1,"[0, 1, 2]","[1, 0, 2]",0
4,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,15,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_15,FAR,True,"[[speaker, top right is green, 1528144581184]]",1,"[{'shapes': [{'color': [329, 14, 50]}, {'color...",2,"[2, 1, 0]","[2, 1, 0]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3630,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,45,6068-c8135454-2875-41d9-918c-60c34fc9460c_45,FAR,True,"[[speaker, Center of middle row is green, cent...",1,"[{'shapes': [{'color': [345, 11, 50]}, {'color...",1,"[2, 0, 1]","[2, 0, 1]",2
3631,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,47,6068-c8135454-2875-41d9-918c-60c34fc9460c_47,FAR,True,"[[speaker, middle tile and middle bottom are b...",1,"[{'shapes': [{'color': [354, 73, 50]}, {'color...",1,"[0, 2, 1]","[0, 1, 2]",1
3632,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,52,6068-c8135454-2875-41d9-918c-60c34fc9460c_52,FAR,True,"[[speaker, top left green, 1526435104682]]",1,"[{'shapes': [{'color': [277, 19, 50]}, {'color...",1,"[2, 0, 1]","[1, 2, 0]",0
3633,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,57,6068-c8135454-2875-41d9-918c-60c34fc9460c_57,FAR,True,"[[speaker, Top middle green, 1526435245005]]",1,"[{'shapes': [{'color': [195, 34, 50]}, {'color...",1,"[2, 1, 0]","[1, 2, 0]",0


# 4. Methods and Dataset

### Patches

In [ ]:
####patches

scorer = BERTScorer(lang="en", rescale_with_baseline=True)

wandb.define_metric("reward/local_step")
wandb.define_metric("reward/*", step_metric="reward/local_step")

def build_prompt(target_position):
    return f"""
Describe the color patch {target_position} so that another model can identify it among three patches.

Rules:
- The other model does not see the patches in the same order as you do.
- Describe only visual properties.
- Mention hue, saturation, brightness, contrast, texture, etc. if useful.
"""

#- Keep it under 30 words.
#- Do not use left, middle, right, center.



def parse_position(text):
    text = text.lower().strip()

    if "left" in text:
        return "left"

    if "middle" in text or "center" in text:
        return "middle"

    if "right" in text:
        return "right"

    return "invalid"



@weave.op()
def judge_patch_from_description(image, description, judge_order):

    judge_prompt = f"""
You see an image with three color patches: left, middle, right.

A description says:
"{description}"

Which patch is described?

Answer only:
left, middle, or right.
"""

    # Hier dein Judge-VLM aufrufen
    img = build_patch_image_with_target_highlight(image, judge_order, **img_kwargs, highlight_target=False)
    #display(img)
    #print(f"Generated Description: {description}")
    answer = run_vlm_judge(img, judge_prompt)
    #print(f"Judge Answer: {answer}")
    #print("-" * 50)

    return  {
        "position": parse_position(answer),
        #"image": img
    }



def run_vlm_judge(image, prompt):
    # Hier rufst du dein Judge-VLM mit dem Bild und Prompt auf und bekommst die Antwort zurück
    # Zum Beispiel:
    # answer = judge_vlm.generate(image=image, prompt=prompt, **generate_kwargs)
    # return answer
    response = judge_model.generate(prompt, image, **generate_kwargs_judge)
    
    # buffer = io.BytesIO()
    # image.save(buffer, format="PNG")
    # img_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    # image_url = f"data:image/png;base64,{img_base64}"

    # response = judge_model.chat.completions.create(
    #     model="Qwen/Qwen3.6-27B-FP8",
    #     messages=[
    #         {
    #             "role": "user",
    #             "content": [
    #                 {
    #                     "type": "image_url",
    #                     "image_url": {
    #                         "url": image_url
    #                     }
    #                 },
    #                 {
    #                     "type": "text",
    #                     "text": prompt
    #                 },
    #             ],
    #         }
    #     ],
    #     temperature=1.0,
    #     top_p=0.95,
    #     presence_penalty=0.0,
    #     max_tokens=800,
    #     extra_body={
    #         "chat_template_kwargs": {
    #             "enable_thinking": False
    #         },
    #         "thinking_budget_tokens": 100,
    #     }
    # )

    # response = response.choices[0].message.content


    
    
    
    
    
    # Dummy-Implementierung für das Beispiel
    return response





def contains_position_words(text):

    forbidden = [
        "left",
        "middle",
        "right",
        "center",
        "centre",
    ]

    return any(w in text.lower() for w in forbidden)






@weave.op()
def compute_reward_bert(description, image, target_position, judge_order, og_description):
    global scorer

    reward = 0.0

    
    bert_weight = 0.5  # Gewichtung für den BERT-Score
    judge_weight = 1-bert_weight

    try:
        F1, _, _ = scorer.score([description], [og_description])


        torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
        t0 = time.perf_counter()
        bert_reward = bert_weight * F1[0].item()
        

        torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
        bert_time = time.perf_counter() - t0
    except Exception as e:
        print(f"Error computing BERT score: {e}")
        bert_reward = 0.0
        bert_time = 0.0

    reward += bert_reward  # F1-Score als Belohnung hinzufügen
        

    torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
    t0 = time.perf_counter()
    result = judge_patch_from_description(
        image=image,
        description=description,
        judge_order=judge_order
    )

    torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
    judge_time = time.perf_counter() - t0

    judge_prediction = result["position"]
    #img = result["image"]

    # Judge richtig?
    prediction_true = judge_prediction == target_position

    if prediction_true:
        judge_reward = 1.0* judge_weight
    else:
        judge_reward = -0.2* judge_weight
    reward += judge_reward

    # Cheating bestrafen
    #if contains_position_words(description):
    #    reward -= 0.5

    # Zu kurz
    word_count = len(description.split())

    # Zu lang
    #reward -= word_count * 0.015

    return {
        "reward": reward,
        "judge_prediction": judge_prediction,
        "word_count": word_count,
        "contains_position_words":
            contains_position_words(description),
        "image": img,
        "bert_reward": bert_reward,
        "judge_reward": judge_reward,
        "prediction_true": prediction_true,
        "bert_time": bert_time,
        "judge_time": judge_time
    }




all_rewards = []
sample_rewards_avg = []
judge_correct_avg = []
word_count_avg = []
contains_position_words_avg = []
reward_local_step = 0




def reward_func(completions, patches, rl_order, judge_order, og_description, **kwargs):
    global all_rewards
    global reward_local_step

    rewards = []
    #print(type(completions[0]))
    #print(completions[0])

    bert_time = 0.0
    judge_time = 0.0
    misc_time = 0.0

    reward_start = time.perf_counter()
    for desc, img, rl_ord, judge_ord, og_desc in zip(
        completions,
        patches,
        rl_order,
        judge_order,
        og_description
    ):
        misc_start = time.perf_counter()
        
        desc = desc[0]["content"]
        
        target_position = None
        if judge_ord[0] == 0:
            target_position = "left"
        elif judge_ord[1] == 0:
            target_position = "middle"
        elif judge_ord[2] == 0:
            target_position = "right"

        misc_time += time.perf_counter() - misc_start

        """result = compute_reward(
            description=desc,
            image=img,
            target_position=target_position,
            judge_order=judge_ord
        )"""

    

        result = compute_reward_bert(
            description=desc,
            image=img,
            target_position=target_position,
            judge_order=judge_ord,
            og_description=og_desc
        )

        bert_time += result["bert_time"]
        judge_time += result["judge_time"]

        rewards.append(result["reward"])

        reward_local_step += 1

        wandb.log({
            "reward/local_step": 
                reward_local_step,
                
            "reward/sample_reward":
                result["reward"],

            "reward/judge_correct":
                int(result["judge_prediction"] == target_position),

            "reward/word_count":
                result["word_count"],

            "reward/contains_position_words":
                int(result["contains_position_words"])
        })

    torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
    reward_total = time.perf_counter() - reward_start

    print(
        f"\nReward total: {reward_total:.2f}s | "
        f"BERT: {bert_time:.2f}s | "
        f"Judge: {judge_time:.2f}s | "
        f"Misc: {misc_time:.2f}s"
    )

    all_rewards.append(rewards)
    return rewards


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
####patches batches

scorer = BERTScorer(lang="en", rescale_with_baseline=True)

wandb.define_metric("reward/local_step")
wandb.define_metric("reward/*", step_metric="reward/local_step")

def build_prompt(target_position):
    return f"""
Describe the color patch {target_position} so that another model can identify it among three patches.

Rules:
- The other model does not see the patches in the same order as you do.
- Describe only visual properties.
- Mention hue, saturation, brightness, contrast, texture, etc. if useful.
- Keep it under 30 words.
- Do not use left, middle, right, center.
"""

#- Keep it under 30 words.
#- Do not use left, middle, right, center.



def parse_position(text):
    text = text.lower().strip()

    if "left" in text:
        return "left"

    if "middle" in text or "center" in text:
        return "middle"

    if "right" in text:
        return "right"

    return "invalid"



@weave.op()
def judge_patch_from_description(image, description, judge_order):

    judge_prompt = f"""
You see an image with three color patches: left, middle, right.

A description says:
"{description}"

Which patch is described?

Answer only:
left, middle, or right.
"""

    # Hier dein Judge-VLM aufrufen
    img = build_patch_image_with_target_highlight(image, judge_order, **img_kwargs, highlight_target=False)
    #display(img)
    #print(f"Generated Description: {description}")
    answer = run_vlm_judge(img, judge_prompt)
    #print(f"Judge Answer: {answer}")
    #print("-" * 50)

    return  {
        "position": parse_position(answer),
        #"image": img
    }

def judge_patch_batch(
    images,
    descriptions,
    judge_orders,
    batch_size=4
):

    judge_images = []
    judge_prompts = []

    for image, description, judge_order in zip(
        images,
        descriptions,
        judge_orders
    ):

        prompt = f"""
        You see an image with three color patches: left, middle, right.

        A description says:
        "{description}"

        Which patch is described?

        Answer only:
        left, middle, or right.
        """

        img = build_patch_image_with_target_highlight(
            image,
            judge_order,
            **img_kwargs,
            highlight_target=False
        )

        judge_images.append(img)
        judge_prompts.append(prompt)

    predictions = []

    # Erstmal bewusst in kleinen GPU-Batches
    for start in range(0, len(judge_images), batch_size):

        end = start + batch_size

        answers = judge_model.generate_batch(
            prompts=judge_prompts[start:end],
            images=judge_images[start:end],

            max_new_tokens=5,
            do_sample=False,
        )

        predictions.extend(
            parse_position(answer)
            for answer in answers
        )

    return predictions



def run_vlm_judge(image, prompt):
    # Hier rufst du dein Judge-VLM mit dem Bild und Prompt auf und bekommst die Antwort zurück
    # Zum Beispiel:
    # answer = judge_vlm.generate(image=image, prompt=prompt, **generate_kwargs)
    # return answer
    response = judge_model.generate(prompt, image, **generate_kwargs_judge)
    
    # buffer = io.BytesIO()
    # image.save(buffer, format="PNG")
    # img_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    # image_url = f"data:image/png;base64,{img_base64}"

    # response = judge_model.chat.completions.create(
    #     model="Qwen/Qwen3.6-27B-FP8",
    #     messages=[
    #         {
    #             "role": "user",
    #             "content": [
    #                 {
    #                     "type": "image_url",
    #                     "image_url": {
    #                         "url": image_url
    #                     }
    #                 },
    #                 {
    #                     "type": "text",
    #                     "text": prompt
    #                 },
    #             ],
    #         }
    #     ],
    #     temperature=1.0,
    #     top_p=0.95,
    #     presence_penalty=0.0,
    #     max_tokens=800,
    #     extra_body={
    #         "chat_template_kwargs": {
    #             "enable_thinking": False
    #         },
    #         "thinking_budget_tokens": 100,
    #     }
    # )

    # response = response.choices[0].message.content


    
    
    
    
    
    # Dummy-Implementierung für das Beispiel
    return response





def contains_position_words(text):

    forbidden = [
        "left",
        "middle",
        "right",
        "center",
        "centre",
    ]

    return any(w in text.lower() for w in forbidden)






@weave.op()
def compute_reward_bert(description, image, target_position, judge_order, og_description):
    global scorer

    reward = 0.0

    
    bert_weight = 0.5  # Gewichtung für den BERT-Score
    judge_weight = 1-bert_weight

    try:
        F1, _, _ = scorer.score([description], [og_description])


        torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
        t0 = time.perf_counter()
        bert_reward = bert_weight * F1[0].item()
        

        torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
        bert_time = time.perf_counter() - t0
    except Exception as e:
        print(f"Error computing BERT score: {e}")
        bert_reward = 0.0
        bert_time = 0.0

    reward += bert_reward  # F1-Score als Belohnung hinzufügen
        

    torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
    t0 = time.perf_counter()
    result = judge_patch_from_description(
        image=image,
        description=description,
        judge_order=judge_order
    )

    torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
    judge_time = time.perf_counter() - t0

    judge_prediction = result["position"]
    #img = result["image"]

    # Judge richtig?
    prediction_true = judge_prediction == target_position

    if prediction_true:
        judge_reward = 1.0* judge_weight
    else:
        judge_reward = -0.2* judge_weight
    reward += judge_reward

    # Cheating bestrafen
    #if contains_position_words(description):
    #    reward -= 0.5

    # Zu kurz
    word_count = len(description.split())

    # Zu lang
    #reward -= word_count * 0.015

    return {
        "reward": reward,
        "judge_prediction": judge_prediction,
        "word_count": word_count,
        "contains_position_words":
            contains_position_words(description),
        "image": img,
        "bert_reward": bert_reward,
        "judge_reward": judge_reward,
        "prediction_true": prediction_true,
        "bert_time": bert_time,
        "judge_time": judge_time
    }




all_rewards = []
sample_rewards_avg = []
judge_correct_avg = []
word_count_avg = []
contains_position_words_avg = []
reward_local_step = 0



def reward_func(completions, patches, rl_order, judge_order, og_description, **kwargs):
    global all_rewards
    global reward_local_step

    descriptions = []
    target_positions = []

    for desc, judge_ord in zip(
        completions,
        judge_order
    ):

        desc = desc[0]["content"]

        descriptions.append(desc)

        if judge_ord[0] == 0:
            target_position = "left"

        elif judge_ord[1] == 0:
            target_position = "middle"

        elif judge_ord[2] == 0:
            target_position = "right"

        else:
            target_position = "invalid"

        target_positions.append(
            target_position
        )

    #judge_start = time.perf_counter()

    judge_predictions = judge_patch_batch(
        images=patches,
        descriptions=descriptions,
        judge_orders=judge_order,
        batch_size=8
    )

    #torch.cuda.synchronize()

    #judge_time = (
    #    time.perf_counter()
    #    - judge_start
    #)

    print("BEFORE BERT sync")
    torch.cuda.synchronize()
    print("BEFORE BERT sync OK")
    try:
        _, _, F1 = scorer.score(
            descriptions,
            list(og_description),
            batch_size=4
        )

    except Exception as e:
        print(f"Error computing BERT score: {e}")

        F1 = [0.0] * len(descriptions)

        for i in range(len(descriptions)):
            try:
                _, _, F1_i = scorer.score(
                    [descriptions[i]],
                    [og_description[i]]
                )

                F1[i] = float(F1_i[0])

            except Exception as e:
                print(
                    f"Error computing BERT score "
                    f"for description {i}: {e}"
                )

                F1[i] = 0.0

    print("AFTER BERT call")

    torch.cuda.synchronize()

    print("AFTER BERT sync OK")

    rewards = []

    bert_weight = 0
    judge_weight = 1

    prediction_correct_list =[]
    word_count_list = []
    contains_position_words_list = []
    bert_reward_list = []
    judge_reward_list = []
    for i in range(len(descriptions)):

        word_count = len(descriptions[i].split())
        word_count_list.append(word_count)

        contains_position_words_list.append(contains_position_words(descriptions[i]))

        bert_reward = bert_weight * float(F1[i])

        correct = (
            judge_predictions[i]
            == target_positions[i]
        )

        if correct:
            judge_reward = (
                1.0 * judge_weight
            )
            prediction_correct_list.append(1)
        else:
            judge_reward = (
                -0.2 * judge_weight
            )
            prediction_correct_list.append(0)

        bert_reward_list.append(bert_reward)
        judge_reward_list.append(judge_reward)
        
        reward = (
            bert_reward +
            judge_reward
        )

        rewards.append(reward)

    reward_local_step += 1

    wandb.log({
        "reward/local_step": 
            reward_local_step,
    
        "reward/sample_reward":
            np.mean(rewards),
    
        "reward/judge_correct":
            np.mean(prediction_correct_list),
    
        "reward/word_count":
            np.mean(word_count_list),
    
        "reward/contains_position_words":
            np.mean(contains_position_words_list)
    })

    for i in range(len(descriptions)):
        image = build_patch_image_with_target_highlight(patches[i], judge_order[i], **img_kwargs, highlight_target=False)
        log_params(
            description=descriptions[i],
            target_position=target_positions[i],
            speaker_order=rl_order[i],
            judge_order=judge_order[i],
            og_description=og_description[i],
            reward=rewards[i],
            judge_prediction=judge_predictions[i],
            word_count=word_count_list[i],
            img=image,
            bert_reward=bert_reward_list[i],
            judge_reward=judge_reward_list[i],
            prediction_true=bool(prediction_correct_list[i]))




    all_rewards.append(rewards)
    return rewards

@weave.op()
def log_params(description, target_position, speaker_order, judge_order, og_description, reward, judge_prediction, word_count, img, bert_reward, judge_reward, prediction_true):
    return



# def reward_func(completions, patches, rl_order, judge_order, og_description, **kwargs):
#     global all_rewards
#     global reward_local_step

#     rewards = []
#     #print(type(completions[0]))
#     #print(completions[0])

#     bert_time = 0.0
#     judge_time = 0.0
#     misc_time = 0.0

#     reward_start = time.perf_counter()
#     for desc, img, rl_ord, judge_ord, og_desc in zip(
#         completions,
#         patches,
#         rl_order,
#         judge_order,
#         og_description
#     ):
#         misc_start = time.perf_counter()
        
#         desc = desc[0]["content"]
        
#         target_position = None
#         if judge_ord[0] == 0:
#             target_position = "left"
#         elif judge_ord[1] == 0:
#             target_position = "middle"
#         elif judge_ord[2] == 0:
#             target_position = "right"

#         misc_time += time.perf_counter() - misc_start

#         """result = compute_reward(
#             description=desc,
#             image=img,
#             target_position=target_position,
#             judge_order=judge_ord
#         )"""

    

#         result = compute_reward_bert(
#             description=desc,
#             image=img,
#             target_position=target_position,
#             judge_order=judge_ord,
#             og_description=og_desc
#         )

#         bert_time += result["bert_time"]
#         judge_time += result["judge_time"]

#         rewards.append(result["reward"])

#         reward_local_step += 1

#         wandb.log({
#             "reward/local_step": 
#                 reward_local_step,
                
#             "reward/sample_reward":
#                 result["reward"],

#             "reward/judge_correct":
#                 int(result["judge_prediction"] == target_position),

#             "reward/word_count":
#                 result["word_count"],

#             "reward/contains_position_words":
#                 int(result["contains_position_words"])
#         })

#     torch.cuda.synchronize()  # Ensure all CUDA operations are complete before timing
#     reward_total = time.perf_counter() - reward_start

#     print(
#         f"\nReward total: {reward_total:.2f}s | "
#         f"BERT: {bert_time:.2f}s | "
#         f"Judge: {judge_time:.2f}s | "
#         f"Misc: {misc_time:.2f}s"
#     )

#     all_rewards.append(rewards)
#     return rewards


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
##patches

train_dataset = []
for i, entry in dataset_train.iterrows():
    #print(f"Entry {i}: {entry}")


    if entry.s_order_t_distr1_distr2[0] == 0:
        target_position = "on the left"
    elif entry.s_order_t_distr1_distr2[1] == 0:
        target_position = "in the middle"
    elif entry.s_order_t_distr1_distr2[2] == 0:
        target_position = "on the right"
    img = build_patch_image_with_target_highlight(entry.hls_values, entry.s_order_t_distr1_distr2, **img_kwargs, highlight_target=False)
    prompt = build_prompt(target_position)

    utterances = entry.conversation
    conversation = ""
    for utt in utterances:
        if conversation == "":
            conversation += f"{utt[1]}"
        else:
            conversation += f", {utt[1]}"
    conversation = re.sub(r"[^A-Za-zÄÖÜäöüß,\s]", "", conversation)
    

    
    #print(f"Prompt: {prompt}")
    #display(img)
    sample = {
        "prompt": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image", 
                        "image": img,
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ],

        "patches": entry.hls_values,
        "rl_order": entry.s_order_t_distr1_distr2,
        "judge_order": entry.l_order_t_distr1_distr2,
        "og_description": conversation
    }
    train_dataset.append(sample)

train_dataset

[{'prompt': [{'role': 'user',
    'content': [{'type': 'image',
      'image': <PIL.Image.Image image mode=RGB size=330x110>},
     {'type': 'text',
      'text': '\nDescribe the color patch on the left so that another model can identify it among three patches.\n\nRules:\n- The other model does not see the patches in the same order as you do.\n- Describe only visual properties.\n- Mention hue, saturation, brightness, contrast, texture, etc. if useful.\n- Keep it under 30 words.\n- Do not use left, middle, right, center.\n'}]}],
  'patches': [[285, 50, 86], [219, 50, 48], [89, 50, 42]],
  'rl_order': [0, 1, 2],
  'judge_order': [0, 1, 2],
  'og_description': 'purple'},
 {'prompt': [{'role': 'user',
    'content': [{'type': 'image',
      'image': <PIL.Image.Image image mode=RGB size=330x110>},
     {'type': 'text',
      'text': '\nDescribe the color patch on the right so that another model can identify it among three patches.\n\nRules:\n- The other model does not see the patches in the

In [15]:
print(judge_model.model.config._attn_implementation)

sdpa


### Grids

In [8]:
####grids

scorer = BERTScorer(lang="en", rescale_with_baseline=True)

wandb.define_metric("reward/local_step")
wandb.define_metric("reward/*", step_metric="reward/local_step")

def build_prompt(target_position):
    return f"""
Describe the color grid {target_position} so that another model can identify it among three grids.

Rules:

- You may use positional word to reference colors in the grid, but not to reference the grid itself among the three grids.
- Describe only visual properties.
- Mention hue, saturation, brightness, contrast, texture if useful.
- Keep it under 60 words.
"""



def parse_position(text):
    text = text.lower().strip()

    if "left" in text:
        return "left"

    if "middle" in text or "center" in text:
        return "middle"

    if "right" in text:
        return "right"

    return "invalid"



@weave.op()
def judge_grid_from_description(image, description, judge_order):

    judge_prompt = f"""
You see an image with three color grids: left, middle, right.

A description says:
"{description}"

Which grid is described?

Answer only:
left, middle, or right.
"""

    # Hier dein Judge-VLM aufrufen
    #print(f"Judge Order in judge_grid_method: {judge_order}")
    img = build_grid_image(image, judge_order, **img_kwargs)
    #display(img)
    #display(img)
    #print(f"Generated Description: {description}")
    answer = run_vlm_judge(img, judge_prompt)
    #print(f"Judge Answer: {answer}")
    #print("-" * 50)

    return  {
        "position": parse_position(answer),
        "image": img,
    }


# def run_vlm_judge(image, prompt):
#     # Hier rufst du dein Judge-VLM mit dem Bild und Prompt auf und bekommst die Antwort zurück
#     # Zum Beispiel:
#     # answer = judge_vlm.generate(image=image, prompt=prompt, **generate_kwargs)
#     # return answer
#     response = judge_model.generate(prompt, image, **generate_kwargs_judge)
#     # Dummy-Implementierung für das Beispiel
#     return response


def run_vlm_judge(image, prompt):
    # Hier rufst du dein Judge-VLM mit dem Bild und Prompt auf und bekommst die Antwort zurück
    # Zum Beispiel:
    # answer = judge_vlm.generate(image=image, prompt=prompt, **generate_kwargs)
    # return answer
    #response = judge_model.generate(prompt, image, **generate_kwargs_judge)
    
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    img_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    image_url = f"data:image/png;base64,{img_base64}"

    response = judge_model.chat.completions.create(
        model="Qwen/Qwen3.8-27B-FP8",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_url
                        }
                    },
                    {
                        "type": "text",
                        "text": prompt
                    },
                ],
            }
        ],
        #temperature=0.7,
        #max_tokens=800,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            },
            "thinking_budget_tokens": 40
        }
    )

    response = response.choices[0].message.content


    
    
    
    
    
    # Dummy-Implementierung für das Beispiel
    return response





def contains_position_words(text):

    forbidden = [
        "left",
        "middle",
        "right",
        "center",
        "centre",
    ]

    return any(w in text.lower() for w in forbidden)






@weave.op()
def compute_reward(description, image, target_position, judge_order):

    reward = 0.0

    judge_prediction = judge_grid_from_description(
        image=image,
        description=description,
        judge_order=judge_order
    )

    # Judge richtig?
    if judge_prediction == target_position:
        reward += 1.0
    else:
        reward -= 0.2

    # Cheating bestrafen
    #if contains_position_words(description):
    #    reward -= 0.5

    # Zu kurz
    word_count = len(description.split())

    # Zu lang
    #reward -= word_count * 0.015

    return {
        "reward": reward,
        "judge_prediction": judge_prediction,
        "word_count": word_count,
        "contains_position_words":
            contains_position_words(description)
    }






@weave.op()
def compute_reward_bert(description, image, target_position, judge_order, og_description):
    global scorer

    reward = 0.0

    F1, _, _ = scorer.score([description], [og_description])

    bert_weight = 0.5  # Gewichtung für den BERT-Score
    judge_weight = 1-bert_weight

    bert_reward = bert_weight * F1[0].item()
    reward += bert_reward  # F1-Score als Belohnung hinzufügen

    result = judge_grid_from_description(
        image=image,
        description=description,
        judge_order=judge_order
    )

    judge_prediction = result["position"]
    img = result["image"]

    


    # Judge richtig?
    prediction_true = judge_prediction == target_position

    if prediction_true:
        judge_reward = 1.0* judge_weight
    else:
        judge_reward = -0.2* judge_weight
    reward += judge_reward

    # Cheating bestrafen
    #if contains_position_words(description):
    #    reward -= 0.5

    # Zu kurz
    word_count = len(description.split())

    # Zu lang
    #reward -= word_count * 0.015

    return {
        "reward": reward,
        "judge_prediction": judge_prediction,
        "word_count": word_count,
        "contains_position_words":
            contains_position_words(description),
        "image": img,
        "bert_reward": bert_reward,
        "judge_reward": judge_reward,
        "prediction_true": prediction_true
    }




all_rewards = []
sample_rewards_avg = []
judge_correct_avg = []
word_count_avg = []
contains_position_words_avg = []
reward_local_step = 0




def reward_func(completions, grids, rl_order, judge_order, og_description, target, **kwargs):
    global all_rewards
    global reward_local_step

    rewards = []
    #print(type(completions[0]))
    #print(completions[0])
    for desc, img, rl_ord, judge_ord, og_desc, tgt in zip(
        completions,
        grids,
        rl_order,
        judge_order,
        og_description,
        target
    ):
        desc = desc[0]["content"]
        target_position = None
        if judge_ord[0] == tgt:
            target_position = "left"
        elif judge_ord[1] == tgt:
            target_position = "middle"
        elif judge_ord[2] == tgt:
            target_position = "right"



        """result = compute_reward(
            description=desc,
            image=img,
            target_position=target_position,
            judge_order=judge_ord
        )"""

        result = compute_reward_bert(
            description=desc,
            image=img,
            target_position=target_position,
            judge_order=judge_ord,
            og_description=og_desc
        )

        #display(result["image"])
        #print(f"OG Description: {og_desc}")
        #print(f"Description: {desc}")
        #print(f"Target: {target_position}")
        #print(f"RL Order: {rl_ord}")
        #print(f"Judge Order: {judge_ord}")
        #print(f"Judge Prediction: {result['judge_prediction']}")
        
        #print("___________________________________________________________________")

        rewards.append(result["reward"])

        reward_local_step += 1

        wandb.log({
            "reward/local_step": 
                reward_local_step,
                
            "reward/sample_reward":
                result["reward"],

            "reward/judge_correct":
                int(result["judge_prediction"] == target_position),

            "reward/word_count":
                result["word_count"],

            "reward/contains_position_words":
                int(result["contains_position_words"])
        })
    all_rewards.append(rewards)
    return rewards


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
##grids

train_dataset = []
for i, entry in dataset_train.iterrows():
    #print(f"Entry {i}: {entry}")



    target = entry.target
    if entry.speaker_order[0] == target:
        target_position = "on the left"
    elif entry.speaker_order[1] == target:
        target_position = "in the middle"
    elif entry.speaker_order[2] == target:
        target_position = "on the right"
    img = build_grid_image(entry.objs, entry.speaker_order, **img_kwargs)
    prompt = build_prompt(target_position)

    utterances = entry.utterances
    conversation = ""
    for utt in utterances:
        if conversation == "":
            conversation += f"{utt[1]}"
        else:
            conversation += f", {utt[1]}"
    conversation = re.sub(r"[^A-Za-zÄÖÜäöüß,\s]", "", conversation)
    

    
    #print(f"Prompt: {prompt}")
    #display(img)
    sample = {
        "prompt": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image", 
                        "image": img,
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ],

        "grids": entry.objs,
        "rl_order": entry.speaker_order,
        "judge_order": entry.listener_order,
        "og_description": conversation,
        "target": target
    }
    train_dataset.append(sample)

train_dataset

[{'prompt': [{'role': 'user',
    'content': [{'type': 'image',
      'image': <PIL.Image.Image image mode=RGB size=1110x370>},
     {'type': 'text',
      'text': '\nDescribe the color grid in the middle so that another model can identify it among three grids.\n\nRules:\n\n- You may use positional word to reference colors in the grid, but not to reference the grid itself among the three grids.\n- Describe only visual properties.\n- Mention hue, saturation, brightness, contrast, texture if useful.\n- Keep it under 60 words.\n'}]}],
  'grids': [{'shapes': [{'color': [284, 23, 50]},
     {'color': [79, 20, 50]},
     {'color': [46, 44, 50]},
     {'color': [317, 3, 50]},
     {'color': [344, 27, 50]},
     {'color': [187, 28, 50]},
     {'color': [72, 71, 50]},
     {'color': [156, 93, 50]},
     {'color': [70, 77, 50]}],
    'gridDimension': 3,
    'cellLength': 40},
   {'shapes': [{'color': [212, 26, 50]},
     {'color': [271, 95, 50]},
     {'color': [237, 8, 50]},
     {'color': [82,

# 5. Training

In [ ]:


training_args = GRPOConfig(

    output_dir="./qwen2vl_patch_grpo",

    report_to="wandb",

    logging_steps=1,

    learning_rate=1e-6,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=2,

    num_generations=8,

    max_completion_length=80,

    num_train_epochs=1,

    bf16=True,

    save_steps=200,

    gradient_checkpointing=False,
)

In [13]:
class TimedGRPOTrainer(GRPOTrainer):

    def _generate_and_score_completions(self, inputs):

        t0 = time.perf_counter()

        result = super()._generate_and_score_completions(inputs)

        t1 = time.perf_counter()

        #print(
        #    f"Generate + Reward + Scoring: "
        #    f"{t1 - t0:.2f}s"
        #)

        return result

trainer = TimedGRPOTrainer(
    model=hf_model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=reward_func,
    processing_class=processor,
)


trainer.train()
wandb.finish()
weave.finish

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


BEFORE BERT sync
BEFORE BERT sync OK
AFTER BERT call
AFTER BERT sync OK


/home/alesau/miniconda3/envs/mllm/lib/python3.14/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self._baseline_vals = torch.from_numpy(
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-0628-7ba5-8564-bd29c4414a59
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-0606-7b2e-96bd-dc68efe3a95c
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-0612-7a7e-95fb-6db4d38ee082
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-064f-7abb-b943-ccdeec6164f7
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-0643-7d1c-b743-f7aa5bd7caca
weave: 🍩 http

Step,Training Loss
1,0.202487
2,0.446867
3,-0.186090
4,-0.256119
5,0.179988
6,-0.037426
7,-0.185325
8,-0.053919
9,-0.085024
10,0.085024


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-3773-70c2-abbc-314cbe8c8ca3
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-3779-7fa1-8ef0-0f3aae78aeb0
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-3782-7dbe-a6d9-569afdd7d06a
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-3788-74aa-8ba8-761c13fcc23c
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-378d-7964-ab7b-5d549b3b3a5e
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-3795-7bd0-8b3c-b97a056bedfb
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-379d-7386-a5bb-3e98085c6280
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f15e-37a4-7784-8753-a48671b13ec1


...

BEFORE BERT sync
BEFORE BERT sync OK
AFTER BERT call
AFTER BERT sync OK


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f210-bd00-76f3-8022-bab6d409ac07
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f210-bd04-7be7-a950-8d4b51581a3a
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f210-bd0a-7bf1-b98a-3741cf26b448
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f210-bd0f-7301-8476-61820646c848
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f210-bd16-752c-93e4-49e7b9fed2eb
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f210-bd1c-7ff6-bd88-aa46521d0427
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f210-bd22-7028-b547-7e44033f23a5
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a0f210-bd27-7684-975e-cf73e811c147


BEFORE BERT sync
BEFORE BERT sync OK
AFTER BERT call
AFTER BERT sync OK


profiling/Time taken: TimedGRPOTrainer._calculate_rewards,▄▂▃▃▂▂▃▃▃▂▂▂▁▂▃▃▃▃▃▂▃▃▃▄▃▃▃▄▂▃▃▃█▄▃▅▂▂▃▃
profiling/Time taken: TimedGRPOTrainer._get_per_token_logps_and_entropies,▂▃▁▁▃▂▅▁▁▂▁▂▁█▅▅▁▃▃▂▂▂▂▁▂▂▁▂▂▂▄▂▄▂▃▂▃▃▁▂
profiling/Time taken: TimedGRPOTrainer._prepare_inputs,▄▅▁▁▁▁▆▅▅▆▁▁█▇▁▁▁▅▁▁▁▁▄▁▁▅▇█▆▁▁▄▇▁▁▁▁▁▄▁
profiling/Time taken: TimedGRPOTrainer.compute_loss,▁▇▂▂▃▄▂▄▂▆▂▁▃▁▂▂▁▂▂▂▅▁▃▁▃▂▂▂▂██▆▇▂▆█▃▂▂▃
profiling/Time taken: TimedGRPOTrainer.reward_func,▄▄▄▄▄▆▁▅▄▅▄▄▅▃▃▅▃▆▄▄▅▄▃▄▄▄▄▄▄▅▄▁▄▅▄▆█▆▅▅
profiling/Time taken: TimedGRPOTrainer.transformers.generate,▂▆▁▅▄▇▄▆▄▂▃▂▄▃▆▁▅▃▂▃▄▅▄▅▂▃▂▃▄▂▇█▂▂▃▅▄▇▁▃
reward/contains_position_words,▇▂▇▆▃█▇▅▃▇▇█▆▆▇▆▅▇▅▆▇▇▇█▅▆▁▆▆▇█▇█▆▅▆▃▇▅█
reward/judge_correct,▅▆▆▆▆█▇▃▅▃▂█▅▁▃▆▅▆▁▆▃▅█▁▃█▃▆▆▃▂▆▂▅▂▃▅█▆▆
reward/local_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇█████
reward/sample_reward,▅▅▃▅▆▆▂▆▅▆█▃▄▆▂█▆█▇▅▁▇▂▅▆▅▇▅▅▂▅▅▇▇█▆▃▆▇▇
+26,...


<function weave.trace.api.finish() -> 'None'>

In [10]:
#GRID
#3.5-4B JUDGE   10:47:56

#Patches
#3-5-9B JUDGE   ca. 40h

trainer = GRPOTrainer(
    model=hf_model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=reward_func,
    processing_class=processor,
)

trainer.train()

wandb.finish()
weave.finish()

NameError: name 'training_args' is not defined

In [15]:
wandb.finish()
weave.finish()

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f152ae746e0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f14445199c0, raw_cell="wandb.finish()
weave.finish()" transformed_cell="wandb.finish()
weave.finish()
" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/alesau/mllm/mllm/train_model.ipynb#X44sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f152ae746e0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f1444519a70, execution_count=15 error_before_exec=None error_in_exec=Connection lost info=<ExecutionInfo object at 7f14445199c0, raw_cell="wandb.finish()
weave.finish()" transformed_cell="wandb.finish()
weave.finish()
" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/alesau/mllm/mllm/train_model.ipynb#X44sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [19]:
print("Judge devices:", {p.device for p in judge_model.model.parameters()})
print("RL devices:", {p.device for p in hf_model.parameters()})

print(
    "allocated:",
    torch.cuda.memory_allocated() / 1024**3,
    "GB"
)

print(
    "reserved:",
    torch.cuda.memory_reserved() / 1024**3,
    "GB"
)

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f39000b86e0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f380cfc1ed0, raw_cell="print("Judge devices:", {p.device for p in judge_m.." transformed_cell="print("Judge devices:", {p.device for p in judge_m.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/alesau/mllm/mllm/train_model.ipynb#X61sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

Judge devices: {device(type='cuda', index=0)}
RL devices: {device(type='cuda', index=0)}
allocated: 13.657028198242188 GB
reserved: 14.025390625 GB
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f39000b86e0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f380cfc2750, execution_count=19 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f380cfc1ed0, raw_cell="print("Judge devices:", {p.device for p in judge_m.." transformed_cell="print("Judge devices:", {p.device for p in judge_m.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/alesau/mllm/mllm/train_model.ipynb#X61sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

# Rest

In [26]:
dataset = train_dataset[12]
img_rl = build_grid_image(dataset["grids"], dataset["rl_order"], **img_kwargs)
img_judge = build_grid_image(dataset["grids"], dataset["judge_order"], **img_kwargs)
display(img_rl)
print("RL Order:", dataset["rl_order"])
display(img_judge)
print("Judge Order:", dataset["judge_order"])
print("OG Description:", dataset["og_description"])

KeyError: 'grids'

In [ ]:
#!pip install bert_score

scorer = BERTScorer(lang="en", rescale_with_baseline=True)



Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:


dataset_train.iloc[0].conversation[0][1]

for i in range(130, 140):
    utterances = dataset_train.iloc[i].conversation
    conversation = ""
    for utt in utterances:
        if conversation == "":
            conversation += f"{utt[1]}"
        else:
            conversation += f", {utt[1]}"
    conversation = re.sub(r"[^A-Za-zÄÖÜäöüß,\s]", "", conversation)
    print(conversation)
    #print("Target: ", dataset_train.iloc[i].target)

yellow
orange
blue
purple
Target is the orange square
purple
tan olive
green
gray
green


In [ ]:
scorer_raw = BERTScorer(lang="en", rescale_with_baseline=False)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
P, R, F1 = scorer.score(
    ["red", "red", "red", "red", "red", "red", "red", "red"],
    ["The red one", "pink", "red", "blue", "green", "red red", "red red red", "the horse rider falls on his back"],
)

print(F1)
print(P)
print(R)
#print()

#P, R, F1 = scorer_raw.score(
#    ["red", "red", "red", "red", "red", "red", "red"],
#    ["The red one", "pink", "red", "blue", "green", "red red", "red red red"],
#)

#print(F1)
#print(F1_new)
#F1[0].item()

tensor([0.2657, 0.4357, 1.0000, 0.6542, 0.5561, 0.7249, 0.5524, 0.0383])
tensor([0.3139, 0.4975, 1.0000, 0.6536, 0.5553, 0.7738, 0.7098, 0.2696])
tensor([ 0.2160,  0.3734,  1.0000,  0.6536,  0.5554,  0.6759,  0.4022, -0.1761])


In [ ]:
!pip install sentence_transformers


model = SentenceTransformer("all-mpnet-base-v2")

cands = ["red"] * 7
refs = ["The red one", "pink", "red", "blue", "green", "red red", "red red red"]

emb_c = model.encode(cands, convert_to_tensor=True)
emb_r = model.encode(refs, convert_to_tensor=True)

scores = util.cos_sim(emb_c, emb_r).diagonal()
print(scores)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tensor([0.5746, 0.7402, 1.0000, 0.7394, 0.7460, 0.7848, 0.7123],
       device='cuda:0')


-uni VPN einrichten
-qwen evaluation (vs. Claude vs. Gemini vs. gemma4)

